# aiomfac_py Colab Bootstrap

This notebook installs **aiomfac_py**, a pure-Python port of the [AIOMFAC](https://github.com/andizuend/AIOMFAC) thermodynamic activity-coefficient model for inorganic–organic aerosol mixtures, in a Google Colab runtime.

Unlike [SeoulTechPSE/fenicsx-colab](https://github.com/SeoulTechPSE/fenicsx-colab), this package has **no compiled dependencies of its own** (pure Python + numpy, with two small optional extras), so there is no conda/micromamba environment and no `%%` cell magic here — a single `pip install` does it.

### What this notebook does
1. Clone the `aiomfac-python` repository (idempotent — safe to re-run)
2. Run `setup_aiomfac.py`, which `pip install`s the package (editable, so you can also edit files under `src/` in this Colab session)
3. Verify the installation

See `01_quickstart.ipynb` in this same folder for worked usage examples once setup is done.

### 1️⃣ Clone Repository (Idempotent)

In [ ]:
# --------------------------------------------------
# 1️⃣ Clone aiomfac-python repository (idempotent)
# --------------------------------------------------
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/SeoulTechPSE/aiomfac-python.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "aiomfac-python"

def run(cmd):
    subprocess.run(cmd, check=True)

if not REPO_DIR.exists():
    print("Cloning aiomfac-python...")
    run(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    print("Repository already exists — pulling latest...")
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

### 2️⃣ Run `setup_aiomfac.py`

By default this installs the `carbonate` extra (`scipy`, needed for the bisulfate/bicarbonate dissociation-equilibrium solvers) but **not** the `smiles` extra (`epam.indigo`, needed only for `aiomfac_py.s2as` — SMILES → AIOMFAC subgroups). Set `INSTALL_SMILES = True` below if you want to try the SMILES example in `01_quickstart.ipynb`; it adds ~10–20s to the install.

In [ ]:
# --------------------------------------------------
# 2️⃣ Install aiomfac_py
# --------------------------------------------------
INSTALL_SMILES = False  # <--- set True to also install epam.indigo (aiomfac_py.s2as)
USE_CLEAN = False       # <--- set True to force a clean reinstall

opts = ["--carbonate"]
if INSTALL_SMILES:
    opts.append("--smiles")
if USE_CLEAN:
    opts.append("--clean")

get_ipython().run_line_magic("run", f"{REPO_DIR / 'setup_aiomfac.py'} {' '.join(opts)}")

### 3️⃣ Sanity Check

In [ ]:
# ==================================================
# 3️⃣ Sanity check
# ==================================================
import aiomfac_py as aiomfac
from aiomfac_py import read_input_file, ActivityModel

print(f"aiomfac_py {aiomfac.__version__} imported OK")

case = read_input_file(REPO_DIR / "tests/reference/inputs/input_0001.txt")
model = ActivityModel(case.components)
res = model.evaluate(case.fractions[0], case.T_K[0], case.basis)
print("gamma (neutrals):", res.gamma_neutral)
print("\n\u2705 Setup verified — continue with 01_quickstart.ipynb")